# 🚀 Put Everything Together - Build Your AI-Based Webapp  

**Welcome to the final lesson!** Today, we’ll synthesize the skills you’ve acquired throughout this course to create a **real-world AI-driven web application**. By the end of this session, you’ll build a **book recommendation system** that:  
- Suggests books based on user preferences.  
- Leverages a structured book database.  
- Integrates advanced AI components for personalized interactions.  

---

## 📌 **What You’ll Learn**  
1. **End-to-End Pipeline Design**: From data preprocessing to deployment.  
2. **Streamlit Fundamentals**: Create intuitive web interfaces rapidly.  
3. **Recommendation Systems**: Build algorithms to match user preferences.  
4. **LLM Integration**: Enhance your app with natural language querying and Q&A capabilities.  

---

## 📚 **Project Overview: Book Recommendation Engine**  

### **Step 1: Preprocess Your Book Dataset**  
- Clean, normalize, and structure raw book data (e.g., titles, genres, author info, ratings).  
- Optimize the dataset for efficient database storage and retrieval.  

### **Step 2: Build the Webapp with Streamlit**  
- Design a user-friendly interface for inputting preferences (e.g., liked genres, favorite authors).  
- Display recommendations dynamically.  

### **Step 3: Integrate a Recommendation Algorithm**  
- Implement content-based filtering model.  
- Suggest books based on the user’s purchase history or stated interests.  

### **Step 4: Add LLM-Powered Query Handling**  
- Use a language model (e.g., GPT-4) to parse complex user queries (e.g., “Find novels published after 2010”).  

### **Step 5: Enable LLM-Driven Q&A**  
- Deploy a second LLM layer to answer user questions about specific books (e.g., “What’s the pacing of *The Silent Patient*?”).  

---

## 🌟 **Why This Matters**  
This project ties together **data engineering**, **machine learning**, and **LLM integration**—skills critical for modern AI applications. By the end, you’ll have a portfolio-ready tool that demonstrates your ability to deliver actionable, user-centric AI solutions.  

**Ready to code? Let’s start building!**  

**_______________________________________________________________________________________________________________**

# 🚀 Let's start the practice!

In [1]:
import re
import pandas as pd
import warnings as wn

wn.filterwarnings("ignore")

In [2]:
# Read Books.csv
df = pd.read_csv("Books.csv", sep=";", on_bad_lines="skip")

df

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
0,1959,The Sirens of Titan,Kurt Vonnegut Jr.,Excellent,4.16,genre fiction,1.00,"Amazon Digital Services, Inc.",4077.0
1,2006,The Shifting Fog,Kate Morton,Intermediate,3.93,genre fiction,2.00,"Amazon Digital Services, Inc.",2889.0
2,1959,Hawaii,James A. Michener,Excellent,4.18,genre fiction,2.00,"Amazon Digital Services, Inc.",106.0
3,2011,State of Wonder,Ann Patchett,Intermediate,3.85,genre fiction,2.00,"Amazon Digital Services, Inc.",605.0
4,1968,A Wizard of Earthsea,Ursula K. Le Guin,Intermediate,3.99,genre fiction,3.00,"Amazon Digital Services, Inc.",1569.0
...,...,...,...,...,...,...,...,...,...
1066,1945,The Great Divorce,C.S. Lewis,Excellent,4.28,fiction,9.99,Penguin Group (USA) LLC,3861.0
1067,2003,Dry: A Memoir,Augusten Burroughs,Excellent,4.01,children,9.99,"Amazon Digital Services, Inc.",3915.0
1068,2012,Ø§Ù„ÙÙŠÙ„ Ø§Ù„Ø£Ø²Ø±Ù‚,Ø£Ø­Ù…Ø¯ Ù…Ø±Ø§Ø¯,Intermediate,3.8,nonfiction,9.99,"Amazon Digital Services, Inc.",2889.0
1069,1601,Twelfth Night,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Publishing Year

Requirements for the "Publishing_Year" Column:
- It must exist, it cannot be `None` or `NaN`.
- It must be a numeric value.
- The value must be positive.
- The year cannot exceed the current year.

In [3]:
df_1 = df.copy()

In [4]:
# Step 1: Remove NaN values
# df['Publishing_Year'].dropna()
start_len = len(df)
df = df[df['Publishing_Year'].notna()]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  1


In [5]:
# Step 2: Keep only numeric values

## Find all nonnumeric values
df[~pd.to_numeric(df['Publishing_Year'], errors='coerce').notna()]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
209,"""1953""",Better Homes & Gardens New Cook Book,Better Homes and Gardens,Excellent,4.14,nonfiction,0.99,"Amazon Digital Services, Inc.",4509.0
449,"""1996""",A Fine Balance,Rohinton Mistry,Intermediate,4.34,genre fiction,2.99,Simon and Schuster Digital Sales Inc,380.0
480,1998-1999,Tripwire,Lee Child,Excellent,4.07,genre fiction,2.99,"Amazon Digital Services, Inc.",106.0
588,01/01/12,A Memory of Light,"Robert Jordan, Brandon Sanderson",Famous,4.5,genre fiction,3.99,"Amazon Digital Services, Inc.",2889.0
770,"""2014""",City of Heavenly Fire,Cassandra Clare,Famous,4.48,genre fiction,4.99,"Amazon Digital Services, Inc.",1338.0


In [6]:
## del double quotes (") from values
# df['Publishing_Year'].replace()

In [7]:
df['Publishing_Year']= df['Publishing_Year'].astype(str).str.replace('"', '')

In [8]:
## convert publish year for record number 480 to 1998
df['Publishing_Year'].loc[480] = 1998


# 1998-1999 -> '1998', '1999' -> '1998'
# df['Publishing_Year'].astype(str).str.split('-')[0]

In [9]:
## convert publish year for record number 588 to 2012
df['Publishing_Year'].loc[588] = 2012

In [10]:
## convert to numerics
df['Publishing_Year'] = pd.to_numeric(df['Publishing_Year'], errors='coerce')

In [11]:
# Step 3: Keep only positive values
start_len = len(df)
df = df[df['Publishing_Year'] > 0]

print("Number of rows filtered: ", start_len - len(df))

# df = df[(df['Publishing_Year'] > 0) & (df['Publishing_Year'] <= 2025)]

Number of rows filtered:  6


In [12]:
# Step 4: Keep only values that do not exceed the current year (2025)
start_len = len(df)
df = df[df['Publishing_Year'] <= 2025]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


### Book Name

Requirements:
- The book name must exist (i.e., it cannot be `None` or `NaN`).  
- The value must be a string.  
- It must not contain special characters.

In [13]:
# Step 1: Remove NaN values
start_len = len(df)
df = df[df['Book_Name'].notna()]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  22


In [14]:
# Step 2: Convert to string values
df['Book_Name']= df['Book_Name'].astype(str)

In [15]:
# Step 3: Remove unwanted characters using regex
# Keep only letters, numbers, spaces, and the allowed characters: . , : ; ! ?

#[^a-zA-Z0-9\s\.\,\:\;\!\?]

df['Book_Name'] = df['Book_Name'].str.replace(r'[^a-zA-Z0-9\s\.\,\:\;\!\?]', '', regex=True)

In [16]:
# Function to remove extra spaces "Rain   Man" -> "Rain Man"
def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

# Function to add spaces between lowercase and uppercase characters "RainMan" -> "Rain Man"
def add_spaces_between_lower_upper(text):
    return re.sub(r'(?=[a-z0-9])(?=[A-Z])', ' ', text)

# Apply both functions on Book Name column
df['Book_Name'] = df['Book_Name'].apply(remove_extra_spaces)
df['Book_Name'] = df['Book_Name'].apply(add_spaces_between_lower_upper)

In [17]:
# Step 4: Check records with Book Name lenght less than 4
df[df['Book_Name'].str.len() < 4]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
158,2002,1,"Hiromu Arakawa, Akira Watanabe",Famous,4.49,genre fiction,0.99,"Amazon Digital Services, Inc.",6183.0
161,2010,1,"Hajime Isayama, Sheldon Drzka",Famous,4.42,genre fiction,0.99,"Amazon Digital Services, Inc.",42552.0
300,2003,1,Bisco Hatori,Excellent,4.36,genre fiction,1.99,HarperCollins Publishers,335.0
333,2012,,"Ø£Ø­Ù„Ø§Ù… Ù…Ø³ØªØºØ§Ù†Ù…ÙŠ, Ahlam Mosteghanemi",Intermediate,3.72,nonfiction,10.91,Simon and Schuster Digital Sales Inc,4320.0
469,1999,1,"Natsuki Takaya, Alethea Nibley, Athena Nibley",Intermediate,4.23,nonfiction,2.99,Simon and Schuster Digital Sales Inc,345.0
606,1967,,"Mikhail Bulgakov, Katherine Tiernan O'Connor, ...",Intermediate,4.32,genre fiction,3.99,"Amazon Digital Services, Inc.",1584.0
610,2011,,"Marie KondÅ, Cathy Hirano",Intermediate,3.77,genre fiction,3.99,"Amazon Digital Services, Inc.",5751.0
841,1962,,"Aleksandr Solzhenitsyn, H.T. Willetts",Intermediate,3.94,genre fiction,5.99,Hachette Book Group,108.0
985,1864,,"Fyodor Dostoyevsky, Andrew R. MacAndrew, Ben M...",Excellent,4.17,genre fiction,7.99,Random House LLC,2889.0
994,1957,,"Boris Pasternak, Max Hayward, Manya Harari, Jo...",Excellent,4.03,fiction,7.99,HarperCollins Publishers,107.0


In [18]:
# Step 5: Delete records with Book Name lenght less than 4
df = df[df['Book_Name'].str.len() >= 4]

### Author

- The book name must exist (i.e., it cannot be None or NaN).
- The value must be a string.
- It must not contain special characters.


In [19]:
# Step 1: Remove NaN values
start_len = len(df)
df = df[df['Author'].notna()]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  2


In [20]:
# Step 2: Convert to string values
df['Author']= df['Author'].astype(str)

In [21]:
# Step 3: Remove unwanted characters using regex
# Keep only letters, spaces, and the allowed characters: . , : ;

# Apply both functions on Author
df['Author'] = df['Author'].apply(remove_extra_spaces)
df['Author'] = df['Author'].apply(add_spaces_between_lower_upper)

# Remove space+comma 
df['Author'] = df['Author'].str.replace(' ,', ',', regex=False)

In [22]:
df[df['Author'].str.len() < 4]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold


### Author_Rating

- Should be one of [Intermediate, Excellent, Famous, Novice]


In [23]:
# Filter rows where 'Author_Rating' is in the allowed list
start_len = len(df)
allowed_ratings = ['Intermediate', 'Excellent', 'Famous', 'Novice']

df = df[df['Author_Rating'].isin(allowed_ratings)]
print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


### Book_Average_Rating

- It must be a numeric value or None.
- The value must be positive.


In [24]:
# Step 1: Keep only numeric values

## Find all nonnumeric values
df[~pd.to_numeric(df['Book_Average_Rating'], errors='coerce').notna()]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
224,1988,Batman: The Killing Joke,"Alan Moore, Brian Bolland, Tim Sale",Intermediate,"""4.36""",genre fiction,0.99,Random House LLC,460.0
456,1984,Thinner,"Richard Bachman, Stephen King",Intermediate,3..66,genre fiction,2.99,"Amazon Digital Services, Inc.",555.0
847,2010,Dead in the Family,Charlaine Harris,Intermediate,"""3.88""",genre fiction,6.15,"Amazon Digital Services, Inc.",243.0


In [25]:
## del double quotes (") from values
df['Book_Average_Rating']= df['Book_Average_Rating'].astype(str).str.replace('"', '')

In [26]:
## convert book average rating for record number 456 to 3.66
df['Book_Average_Rating'].loc[456] = 3.66

In [27]:
## convert to numerics
df['Book_Average_Rating'] = pd.to_numeric(df['Book_Average_Rating'], errors='coerce')

In [28]:
# Step 2: filter only positive number
start_len = len(df)
df = df[df['Book_Average_Rating'] >= 0]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


### Genre

- Should be one of [genre fiction, nonfiction, fiction, children]

In [29]:
# Filter rows where 'Genre' is in the allowed list
allowed_genre = ['genre fiction', 'nonfiction', 'fiction', 'children']

start_len = len(df)
df = df[df['Genre'].isin(allowed_genre)]
print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


### Sale_Price

- It must be a numeric value.
- The value must be positive.

In [30]:
# Step 1: Keep only numeric values

## Find all nonnumeric values
df[~pd.to_numeric(df['Sale_Price'], errors='coerce').notna()]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold


In [31]:
## convert to numerics
df['Sale_Price'] = pd.to_numeric(df['Sale_Price'], errors='coerce')

In [32]:
# Step 2: filter only positive number
start_len = len(df)
df = df[df['Sale_Price'] > 0]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  2


## Publisher

- The value must be a string.
- It must not contain special characters.

In [33]:
# Step 1: Convert to string values
df['Publisher']= df['Publisher'].astype(str)

In [34]:
# Step 2: Get all Publisher values
df['Publisher'].value_counts().reset_index().sort_values(by='index')

,index,Publisher
7,Amazon Digital Services,16
0,"Amazon Digital Services, Inc.",565
4,Hachette Book Group,61
8,HarperCollins Christian Publishing,4
3,HarperCollins Publishers,66
9,HarperCollins Publishing,4
6,Macmillan,41
2,Penguin Group (USA) LLC,97
1,Random House LLC,117
5,Simon and Schuster Digital Sales Inc,54


In [35]:
# Step 3: Preprocess Publisher
## Amazon Digital Services,  Inc. -> Amazon Digital Services
df['Publisher'] = df['Publisher'].replace('Amazon Digital Services,  Inc.', 'Amazon Digital Services')

In [36]:
## HarperCollins Christian Publishing -> HarperCollins Publishers
df['Publisher'] = df['Publisher'].replace('HarperCollins Christian Publishing', 'HarperCollins Publishers')

In [37]:
## HarperCollins Publishing -> HarperCollins Publishers
df['Publisher'] = df['Publisher'].replace('HarperCollins Publishing', 'HarperCollins Publishers')

In [38]:
df['Publisher'].value_counts().reset_index().sort_values(by='index')

,index,Publisher
0,Amazon Digital Services,581
4,Hachette Book Group,61
3,HarperCollins Publishers,74
6,Macmillan,41
2,Penguin Group (USA) LLC,97
1,Random House LLC,117
5,Simon and Schuster Digital Sales Inc,54


### Units_Sold

- It must be a numeric value.
- The value must be positive.

In [39]:
# Step 1: Keep only numeric values

## Find all nonnumeric values
df[~pd.to_numeric(df['Units_Sold'], errors='coerce').notna()]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold


In [40]:
## convert to numerics
df['Units_Sold'] = pd.to_numeric(df['Units_Sold'], errors='coerce')

In [41]:
# Step 2: filter only positive number
start_len = len(df)
df = df[df['Units_Sold'] >= 0]

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


In [42]:
df

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
0,1959,The Sirens of Titan,Kurt Vonnegut Jr.,Excellent,4.16,genre fiction,1.00,Amazon Digital Services,4077.0
1,2006,The Shifting Fog,Kate Morton,Intermediate,3.93,genre fiction,2.00,Amazon Digital Services,2889.0
2,1959,Hawaii,James A. Michener,Excellent,4.18,genre fiction,2.00,Amazon Digital Services,106.0
3,2011,State of Wonder,Ann Patchett,Intermediate,3.85,genre fiction,2.00,Amazon Digital Services,605.0
4,1968,A Wizard of Earthsea,Ursula K. Le Guin,Intermediate,3.99,genre fiction,3.00,Amazon Digital Services,1569.0
...,...,...,...,...,...,...,...,...,...
1063,1977,Miss Nelson Is Missing!,"Harry Allard, James Marshall",Excellent,4.26,genre fiction,9.99,Penguin Group (USA) LLC,31320.0
1064,2016,The Nest,Cynthia D'Aprix Sweeney,Novice,3.45,nonfiction,9.99,Amazon Digital Services,4077.0
1065,2012,Deadlocked,Charlaine Harris,Intermediate,3.65,nonfiction,9.99,Amazon Digital Services,106.0
1066,1945,The Great Divorce,C.S. Lewis,Excellent,4.28,fiction,9.99,Penguin Group (USA) LLC,3861.0


### Analyse all rows together

- delete duplicates
- find rows with the same book names which can differ slightly (e.g., due to typos, additional symbols, or formatting)
- find rows with the same autor names which can differ slightly (e.g., due to typos, additional symbols, or formatting)

In [43]:
# Step 1: delete duplicates
start_len = len(df)
df = df.drop_duplicates()

print("Number of rows filtered: ", start_len - len(df))

Number of rows filtered:  0


In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 2: Find similar book names based on TF-IDF (term frequency and inverse document frequency) method 
def find_similarity_in_rows(df, column_name):
    names = df[column_name]

    # Vectorize the text using TF-IDF
    vectorizer = TfidfVectorizer().fit_transform(names)
    similarity_matrix = cosine_similarity(vectorizer)
    
    # calculate similarity
    similarity_df = pd.DataFrame(similarity_matrix)
    
    # filter rows based on similarity
    filtered_rows = similarity_df[(similarity_df >= 0.85) & (similarity_df < 1)].dropna(how='all')
    
    # find similar items
    for i in range(0, len(filtered_rows)):
        first_index = filtered_rows.index[i]
        second_index = filtered_rows.iloc[i].dropna().index[0]
        
        if first_index != second_index:
            print("Similar names: '{}' and '{}'".format(df[column_name].iloc[first_index], 
                                                        df[column_name].iloc[second_index]))

In [45]:
find_similarity_in_rows(df, 'Book_Name')

Similar names: 'Invisible Man' and 'The Invisible Man'
Similar names: 'The Messenger' and 'Messenger'
Similar names: 'The Invisible Man' and 'Invisible Man'
Similar names: 'The Essential Calvin and Hobbes: A Calvin and Hobbes Treasury' and 'Calvin and Hobbes'
Similar names: 'Never Too Far Too Far, 2' and 'Fallen Too Far Too Far, 1'
Similar names: 'Messenger' and 'The Messenger'
Similar names: 'Calvin and Hobbes' and 'The Essential Calvin and Hobbes: A Calvin and Hobbes Treasury'
Similar names: 'Fallen Too Far Too Far, 1' and 'Never Too Far Too Far, 2'


In [46]:
find_similarity_in_rows(df, 'Author')

In [47]:
df[(df['Book_Name'] == 'Invisible Man') | (df['Book_Name'] == 'The Invisible Man')]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
68,1952,Invisible Man,Ralph Ellison,Intermediate,3.84,genre fiction,0.99,Amazon Digital Services,460.0
162,1897,The Invisible Man,H.G. Wells,Intermediate,3.62,genre fiction,0.99,Macmillan,38016.0


In [48]:
df[(df['Book_Name'] == 'Messenger') | (df['Book_Name'] == 'The Messenger')]

,Publishing_Year,Book_Name,Author,Author_Rating,Book_Average_Rating,Genre,Sale_Price,Publisher,Units_Sold
95,2002,The Messenger,Markus Zusak,Excellent,4.09,genre fiction,0.99,Amazon Digital Services,3969.0
615,2004,Messenger,Lois Lowry,Intermediate,3.90,genre fiction,3.99,Amazon Digital Services,30888.0


In [49]:
# Step 3: save file in csv
df.to_csv("Books_preprocessed.csv", index=False)

### Create similarity score matrix for recommendation-system

In [50]:
### Create df_recommendation from df with Hot-One Incoder for years, Author_Rating, Book_Average_Rating, Genre, Sale_Price, Publisher 

In [51]:
# Step 1: Define the number of bins
num_bins = 4  # Create 5 bins

# Step 2: Create bins and assign each value to a class

In [52]:
# Step 3: Create Book_Name___Author column


In [53]:
# Step 4: Select only necessary columns 'Publishing_Year', 'Author_Rating', 'Book_Average_Rating', 'Genre', 'Sale_Price', 'Publisher', 'Book_Name___Author'

In [54]:
# Set 5: Set Book_Name___Author as index

In [55]:
# Apply one-hot encoding to the identified categorical columns

In [56]:
# Calculate cosine similarity

In [57]:
# Save similarity matrix as a csv